In [1]:
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [2]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [3]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in course FAQ."
            }
        }
    },
    "required": ["query"],
    "additionalProperties": False
}

In [5]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [9]:
len(response.output)

1

In [15]:
call = response.output[0]
call.arguments

'{"query":"Can I join the course after it has started? discovered the course can I join it late enrollment"}'

In [13]:
import json

args = json.loads(call.arguments)
args

{'query': 'Can I join the course after it has started? discovered the course can I join it late enrollment'}

In [17]:
results = search(**args)
result_json = json.dumps(results, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course

In [18]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

In [40]:
messages.append(call)
messages.append(function_call_output)


In [41]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course can I join it late enrollment"}', call_id='call_61aZdd0Z7gplPnEfecOMBUVz', name='search', type='function_call', id='fc_0d05bc93f2246a47006a33ca02a47c819f9ee827abe8988f07', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_61aZdd0Z7gplPnEfecOMBUVz',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a se

In [42]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [43]:
print(response.output_text)

Yes — you can still join.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions.


In [72]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Perform one search at a time. Analyse results in between searches. 

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


In [ ]:

question = 'I just discovered the course. Can I join it?'

messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]


In [56]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [60]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)


In [68]:
response.output[0]

ResponseOutputMessage(id='msg_039eefbd65f7e7f0006a33e72055fc8192a48c50651a0d8964', content=[ResponseOutputText(annotations=[], text='Yes — you can still join.\n\nYou can start learning and submitting homework while the submission form is open, even if you just discovered the course. If you want a certificate, make sure you submit your project while they’re still accepting submissions.\n\nIf you want, I can also help you with the best way to start the course or explain the certificate requirements.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [69]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        # process function call
        print('function call: ', item.name, item.arguments)
        function_call_output = make_call(item)
        messages.append(function_call_output)
        
    elif item.type == 'message':
        # do something else
        print('ASSISTANT: ')
        print(item.content[0].text)
        pass

ASSISTANT: 
Yes — you can still join.

You can start learning and submitting homework while the submission form is open, even if you just discovered the course. If you want a certificate, make sure you submit your project while they’re still accepting submissions.

If you want, I can also help you with the best way to start the course or explain the certificate requirements.


In [58]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course enroll discovered course can I join registration late enrollment"}', call_id='call_GGwBrEXC3kmSTXxMGaN9oial', name='search', type='function_call', id='fc_039eefbd65f7e7f0006a33e4249e1c8192a7769e04b699fe0f', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course FAQ enrollment join late registration discovered course"}', 

In [74]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1
    while True:
        print(f'iteration: #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                # process function call
                print('function call: ', item.name, item.arguments)
                function_call_output = make_call(item)
                messages.append(function_call_output)
                has_function_calls = True
                
            elif item.type == 'message':
                # do something else
                print('ASSISTANT: ')
                last_answer = item.content[0].text
                print(last_answer)
                pass
        
        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [84]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Perform one search at a time. Analyse results, compare to original question, and perform new search to find better result. 

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.

The answer must come from the search results. If the answer is not in the search results, do not use your own personal knowledge, and feign ignorance.
""".strip()

question = 'How do i install github?'


In [85]:
agent_loop(instructions,question)

iteration: #1...
function call:  search {"query":"how to install github install github"}
iteration: #2...
ASSISTANT: 
I couldn’t find a course FAQ entry about **installing GitHub** specifically.

If you meant:
- **Git**: that’s the version control tool you install locally.
- **GitHub Desktop**: that’s the app you can install from GitHub.
- **A GitHub account**: that’s created on the GitHub website.

If you want, I can help you find the right one—do you want Git, GitHub Desktop, or a GitHub account?


'I couldn’t find a course FAQ entry about **installing GitHub** specifically.\n\nIf you meant:\n- **Git**: that’s the version control tool you install locally.\n- **GitHub Desktop**: that’s the app you can install from GitHub.\n- **A GitHub account**: that’s created on the GitHub website.\n\nIf you want, I can help you find the right one—do you want Git, GitHub Desktop, or a GitHub account?'

In [87]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [89]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """

    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [90]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [91]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [92]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)


In [93]:

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [101]:
result = runner.loop(
    prompt = 'What is the capital of the UK',
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [100]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nPerform one search at a time. Analyse results, compare to original question, and perform new search to find better result. \n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n\nThe answer must come from the search results. If the answer is not in the search results, do not use your own personal knowledge, and feign ignorance.", role='developer', phase=None, type=None),
 EasyInputMessage(content='What is callback?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"callback definition programming FAQ"}', call_id='call_Ue6X1t7FmRb5ppa3mQT8EJsb'

In [103]:
result2 = runner.loop(
    prompt = 'Sure, look for geography related entries',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [104]:
runner.run()


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nPerform one search at a time. Analyse results, compare to original question, and perform new search to find better result. \n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n\nThe answer must come from the search results. If the answer is not in the search results, do not use your own personal knowledge, and feign ignorance.", role='developer', phase=None, type=None), EasyInputMessage(content='how do I run docker i ndocker?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"how do I run docker in docker docker FAQ"}',